# Лабораторная работа № 4
## Объектно-ориентированное программирование. Принципы SOLID

**Среда выполнения:** Google Colab + GitLab

**Прикладная задача:** «Доменная модель интернет-магазина» — спроектировать объектную модель для управления товарами, клиентами, заказами и скидками, применяя инкапсуляцию, наследование, полиморфизм, магические методы и принципы SOLID.

**Автор:** <ФИО>

**Группа:** <номер группы>

**Дата:** <дата>

---
## Часть I. Теория

### 1. Классы и объекты

**Класс** — это шаблон (чертёж) для создания объектов. **Объект** — экземпляр класса, обладающий своим состоянием и поведением. В Python классы определяются с помощью ключевого слова `class`.

```python
class Product:
    def __init__(self, name: str, price: float) -> None:
        self.name = name
        self.price = price
```

Метод `__init__` — это конструктор, который автоматически вызывается при создании объекта. Параметр `self` — это ссылка на сам экземпляр класса.

### 1.1. Инкапсуляция и свойства (`@property`)

Инкапсуляция — это скрытие внутреннего состояния объекта и предоставление контролируемого доступа к нему. В Python нет строгих модификаторов доступа (как `private` в C++), но есть соглашения:
*   `_name` — защищённый атрибут (protected). К нему не стоит обращаться снаружи.
*   `__name` — приватный атрибут (private). Python искажает его имя (name mangling), чтобы предотвратить случайный доступ.
*   `@property` — позволяет создать метод-геттер, который будет обращаться как обычный атрибут.
*   `@name.setter` — позволяет добавить логику (например, валидацию) при изменении значения.

```python
class Customer:
    def __init__(self, name: str, balance: float = 0.0) -> None:
        self.name = name
        self._balance = balance

    @property
    def balance(self) -> float:
        return self._balance

    @balance.setter
    def balance(self, value: float) -> None:
        if value < 0:
            raise ValueError("Баланс не может быть отрицательным")
        self._balance = value
```

### 1.2. Магические методы

Магические методы (dunder methods, от double underscore) позволяют встроить объекты пользовательских классов в синтаксис языка Python.

| Метод | Вызов | Назначение |
|---|---|---|
| `__str__` | `str(obj)`, `print(obj)` | Читаемое представление для пользователя |
| `__repr__` | `repr(obj)` | Однозначное представление для отладки |
| `__eq__` | `obj1 == obj2` | Сравнение на равенство |
| `__add__` | `obj1 + obj2` | Сложение |
| `__len__` | `len(obj)` | Длина (количество элементов) |

```python
class OrderItem:
    def __init__(self, product: str, quantity: int) -> None:
        self.product = product
        self.quantity = quantity

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, OrderItem):
            return NotImplemented
        return self.product == other.product

    def __add__(self, other: "OrderItem") -> "OrderItem":
        if self.product != other.product:
            raise ValueError("Нельзя складывать разные товары")
        return OrderItem(self.product, self.quantity + other.quantity)
```

### 1.3. Наследование и полиморфизм

**Наследование** позволяет создавать новый класс на основе существующего, переиспользуя его код. Дочерний класс вызывает `super().__init__()` для инициализации родительской части.

**Полиморфизм** — это способность объектов с одинаковым интерфейсом (например, наследников одного базового класса) вести себя по-разному. Вызов метода приводит к выполнению кода, соответствующего конкретному классу объекта.

```python
class Customer:
    def get_discount(self) -> float:
        return 0.0

class VipCustomer(Customer):
    def get_discount(self) -> float:
        return 0.10
```

### 1.4. Абстрактные классы (ABC)

**Абстрактный класс** — это класс, который нельзя инстанцировать (создать его объект). Он служит контрактом для наследников. Если наследник не реализует все абстрактные методы, создать его объект тоже не получится.

```python
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, total: float) -> float:
        ...
```

### 1.5. Принципы SOLID

SOLID — это пять принципов объектно-ориентированного проектирования, делающих систему гибкой и поддерживаемой.

| Принцип | Расшифровка | Суть |
|---|---|---|
| **SRP** | Single Responsibility | У класса должна быть только одна причина для изменения. |
| **OCP** | Open/Closed | Классы открыты для расширения, но закрыты для модификации. |
| **LSP** | Liskov Substitution | Наследник должен без ошибок подставляться вместо родителя. |
| **ISP** | Interface Segregation | Лучше много специализированных интерфейсов, чем один огромный. |
| **DIP** | Dependency Inversion | Зависимость от абстракций, а не от конкретных реализаций. |

---
## Примеры

В качестве прикладной задачи рассмотрим доменную модель интернет-магазина. У нас есть товары, клиенты (с балансом), заказы (состоящие из позиций) и скидки. В будущем заказы нужно будет сохранять в базу данных.

### Пример 1. Инкапсуляция и валидация данных

In [1]:
class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        self.name = name
        self.price = price
        self.stock = stock

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        if value < 0:
            raise ValueError("Цена не может быть отрицательной")
        self._price = value

# Проверка
p = Product("Ноутбук", 75000.0, 10)
print(p.price)
try:
    p.price = -100
except ValueError as e:
    print("Ошибка:", e)

75000.0
Ошибка: Цена не может быть отрицательной


### Пример 2. Наследование и `super()`

In [2]:
class Customer:
    def __init__(self, name: str, balance: float = 0.0) -> None:
        self.name = name
        self.balance = balance

class VipCustomer(Customer):
    def __init__(self, name: str, balance: float = 0.0, bonus: float = 500.0) -> None:
        # Вызов конструктора родителя + добавление бонуса
        super().__init__(name, balance + bonus)
        self.discount_rate = 0.10

vip = VipCustomer("Анна", 1000.0)
print(f"Клиент: {vip.name}, Баланс с бонусом: {vip.balance}, Скидка: {vip.discount_rate}")

Клиент: Анна, Баланс с бонусом: 1500.0, Скидка: 0.1


### Пример 3. Композиция и магические методы

In [3]:
class OrderItem:
    def __init__(self, product: Product, quantity: int) -> None:
        self.product = product
        self.quantity = quantity

    @property
    def subtotal(self) -> float:
        return self.product.price * self.quantity

class Order:
    def __init__(self, customer: Customer) -> None:
        self.customer = customer
        self.items = []
        self.discount = None

    def add_item(self, item: OrderItem) -> None:
        self.items.append(item)

    def total(self) -> float:
        base = sum(i.subtotal for i in self.items)
        if self.discount:
            return self.discount.apply(base)
        return base

    def __len__(self) -> int:
        return len(self.items)

    def __str__(self) -> str:
        return f"Заказ для {self.customer.name}. Позиций: {len(self.items)}. Итого: {self.total()} руб."

### Пример 4. Полиморфизм и принцип OCP (Open/Closed)

In [4]:
from abc import ABC, abstractmethod

class Discount(ABC):
    @abstractmethod
    def apply(self, total: float) -> float: ...

class PercentageDiscount(Discount):
    def __init__(self, percent: float) -> None:
        self.percent = percent

    def apply(self, total: float) -> float:
        return total * (1 - self.percent / 100)

class FixedDiscount(Discount):
    def __init__(self, amount: float) -> None:
        self.amount = amount

    def apply(self, total: float) -> float:
        return max(0, total - self.amount)

# Принцип OCP здесь работает так: чтобы добавить новую скидку,
# мы создаём нового наследника Discount, не меняя код класса Order.

### Пример 5. Принцип DIP (Dependency Inversion)

In [5]:
import uuid

class OrderRepository(ABC):
    @abstractmethod
    def save(self, order: Order) -> str: ...

class InMemoryOrderRepository(OrderRepository):
    def __init__(self) -> None:
        self._storage = {}

    def save(self, order: Order) -> str:
        order_id = str(uuid.uuid4())
        self._storage[order_id] = order
        return order_id

class OrderService:
    # Сервис зависит от абстракции OrderRepository, а не от конкретной реализации
    def __init__(self, repository: OrderRepository) -> None:
        self.repository = repository

    def place_order(self, order: Order) -> str:
        return self.repository.save(order)

# Проверка сквозного сценария
p1 = Product("Книга", 1000.0, 5)
c1 = Customer("Иван", 5000.0)
order = Order(c1)
order.add_item(OrderItem(p1, 2))

repo = InMemoryOrderRepository()
service = OrderService(repo)
order_id = service.place_order(order)
print(order)
print("Заказ сохранен с ID:", order_id)

Заказ для Иван. Позиций: 1. Итого: 2000.0 руб.
Заказ сохранен с ID: 69707b35-4fda-4f2b-a2ff-01d0d82f923e


---
## Часть II. Задания

### Задания на освоение концепции (1–10)

**Задание 1. Базовый класс и магический метод `__str__`.**
Создайте класс `Product` с полями `name: str`, `price: float`, `stock: int`. Реализуйте `__str__` для красивого вывода и `__repr__` для отладки. Проверьте работу методов через `print()` и `repr()`.

In [1]:
class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        # базовая валидация: цена и остаток не могут быть отрицательными
        if price < 0:
            raise ValueError("Цена не может быть отрицательной")
        if stock < 0:
            raise ValueError("Остаток не может быть отрицательным")
        self.name = name
        self.price = price
        self.stock = stock

    def __str__(self) -> str:
        # человекочитаемый вывод для пользователя
        return f"Товар: {self.name}, цена: {self.price:.2f} руб., на складе: {self.stock} шт."

    def __repr__(self) -> str:
        # однозначное представление для отладки: выглядит как код, создающий объект
        return f"Product(name={self.name!r}, price={self.price!r}, stock={self.stock!r})"


# Проверка
p = Product("Ноутбук", 75999.99, 12)
print(p)        # вызывает __str__
print(repr(p))  # вызывает __repr__

Товар: Ноутбук, цена: 75999.99 руб., на складе: 12 шт.
Product(name='Ноутбук', price=75999.99, stock=12)


**Задание 2. Инкапсуляция и свойства (`@property`).**
Создайте класс `Customer` с полями `name: str` и `_balance: float`. Реализуйте `balance` как свойство только для чтения. Добавьте методы `deposit(amount)` и `withdraw(amount)`. При попытке списать больше, чем есть, поднимайте `ValueError`.

In [2]:
class Customer:
    def __init__(self, name: str, balance: float = 0.0) -> None:
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")
        self.name = name
        self._balance = balance

    @property
    def balance(self) -> float:
        # только чтение: сеттера нет, снаружи баланс не изменить
        return self._balance

    def deposit(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма пополнения должна быть положительной")
        self._balance += amount

    def withdraw(self, amount: float) -> None:
        if amount <= 0:
            raise ValueError("Сумма списания должна быть положительной")
        if amount > self._balance:
            raise ValueError(
                f"Недостаточно средств: запрошено {amount}, доступно {self._balance}"
            )
        self._balance -= amount

    def __repr__(self) -> str:
        return f"Customer(name={self.name!r}, balance={self._balance!r})"


# Проверка
c = Customer("Иван", 1000.0)
print(c.name, c.balance)   # Иван 1000.0

c.deposit(500)
print(c.balance)           # 1500.0

c.withdraw(200)
print(c.balance)           # 1300.0

try:
    c.withdraw(5000)
except ValueError as e:
    print("Ошибка:", e)    # Ошибка: Недостаточно средств: ...

try:
    c.balance = 999        # свойство только для чтения
except AttributeError as e:
    print("Ошибка:", e)    # property 'balance' of 'Customer' object has no setter

Иван 1000.0
1500.0
1300.0
Ошибка: Недостаточно средств: запрошено 5000, доступно 1300.0
Ошибка: property 'balance' of 'Customer' object has no setter


**Задание 3. Магические методы сравнения и сложения.**
Создайте класс `OrderItem` (товар + количество). Реализуйте `__eq__` для сравнения позиций по товару и `__add__` для сложения двух позиций одного товара (увеличение количества). Проверьте: `OrderItem("Мышь", 1) + OrderItem("Мышь", 2)`.

In [5]:
class OrderItem:
    def __init__(self, product: str, quantity: int) -> None:
        if quantity <= 0:
            raise ValueError("Количество должно быть положительным")
        self.product = product
        self.quantity = quantity

    def __eq__(self, other: object) -> bool:
        # сравниваем позиции только по товару, количество не учитываем
        if not isinstance(other, OrderItem):
            return NotImplemented
        return self.product == other.product

    def __add__(self, other: "OrderItem") -> "OrderItem":
        # складывать можно только позиции одного товара
        if not isinstance(other, OrderItem):
            return NotImplemented
        if self.product != other.product:
            raise ValueError(
                f"Нельзя сложить разные товары: {self.product!r} и {other.product!r}"
            )
        return OrderItem(self.product, self.quantity + other.quantity)

    def __repr__(self) -> str:
        return f"OrderItem({self.product!r}, {self.quantity})"


# Проверка
a = OrderItem("Мышь", 1)
b = OrderItem("Мышь", 2)
c = OrderItem("Клавиатура", 1)

print(a + b)              # OrderItem('Мышь', 3)
print(a == b)             # True  (один и тот же товар)
print(a == c)             # False (разные товары)

try:
    a + c
except ValueError as e:
    print("Ошибка:", e)   # Ошибка: Нельзя сложить разные товары: 'Мышь' и 'Клавиатура'

OrderItem('Мышь', 3)
True
False
Ошибка: Нельзя сложить разные товары: 'Мышь' и 'Клавиатура'


**Задание 4. Наследование и `super()`.**
Создайте класс `VipCustomer(Customer)`, который при создании получает бонусный баланс, а метод `get_discount` возвращает 10%. Используйте `super()` для вызова конструктора родителя.

In [7]:
# ============================================================
# ЗАДАНИЕ 4: Наследование и super()
# ============================================================

class VipCustomer(Customer):
    def __init__(self, name: str, balance: float = 0.0, bonus: float = 0.0) -> None:
        if bonus < 0:
            raise ValueError("Бонус не может быть отрицательным")
        # super() — вызываем конструктор Customer (задание 2)
        super().__init__(name, balance + bonus)
        self.bonus = bonus

    def get_discount(self) -> float:
        # VIP получает 10%, в отличие от обычного Customer (0%)
        return 10.0

    def __repr__(self) -> str:
        return (
            f"VipCustomer(name={self.name!r}, "
            f"balance={self._balance!r}, bonus={self.bonus!r})"
        )


# ============================================================
# Проверка (без Order)
# ============================================================

v = VipCustomer("Мария", 1000.0, bonus=500.0)
print(v.name, v.balance, v.get_discount())      # Мария 1500.0 10.0

# Унаследованные методы из задания 2 работают
v.deposit(200)
v.withdraw(300)
print(v.balance)                                # 1400.0

try:
    v.withdraw(99999)
except ValueError as e:
    print("Ошибка:", e)                         # Недостаточно средств...

print(repr(v))

Мария 1500.0 10.0
1400.0
Ошибка: Недостаточно средств: запрошено 99999, доступно 1400.0
VipCustomer(name='Мария', balance=1400.0, bonus=500.0)


**Задание 5. `@classmethod` и `@staticmethod`.**
Добавьте в `Product`:
1. `@classmethod` `from_dict(cls, data: dict) -> Product` для создания товара из словаря.
2. `@staticmethod` `is_valid_price(price: float) -> bool` для проверки, что цена положительная.

In [9]:
# ============================================================
# ЗАДАНИЕ 5: @classmethod и @staticmethod в Product
# ============================================================

class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        if not self.is_valid_price(price):
            raise ValueError("Цена должна быть положительной")
        if stock < 0:
            raise ValueError("Остаток не может быть отрицательным")
        self.name = name
        self.price = price
        self.stock = stock

    def __str__(self) -> str:
        return f"Товар: {self.name}, цена: {self.price:.2f} руб., на складе: {self.stock} шт."

    def __repr__(self) -> str:
        return f"Product(name={self.name!r}, price={self.price!r}, stock={self.stock!r})"

    # 1. @classmethod — альтернативный конструктор из словаря
    @classmethod
    def from_dict(cls, data: dict) -> "Product":
        # cls — это сам класс (Product или его наследник), а не экземпляр
        return cls(
            name=data["name"],
            price=data["price"],
            stock=data["stock"],
        )

    # 2. @staticmethod — утилита, не нужен ни cls, ни self
    @staticmethod
    def is_valid_price(price: float) -> bool:
        return isinstance(price, (int, float)) and price > 0


# ============================================================
# Проверка
# ============================================================

# from_dict: создание из словаря
data = {"name": "Ноутбук", "price": 75999.99, "stock": 12}
p = Product.from_dict(data)
print(p)                        # Товар: Ноутбук, цена: 75999.99 руб., на складе: 12 шт.
print(repr(p))                  # Product(name='Ноутбук', price=75999.99, stock=12)

# is_valid_price: статическая проверка без создания объекта
print(Product.is_valid_price(100))    # True
print(Product.is_valid_price(0))      # False
print(Product.is_valid_price(-5))     # False
print(Product.is_valid_price("abc"))  # False

# Валидация в __init__ использует тот же статический метод
try:
    Product("Брак", -10, 1)
except ValueError as e:
    print("Ошибка:", e)         # Ошибка: Цена должна быть положительной

Товар: Ноутбук, цена: 75999.99 руб., на складе: 12 шт.
Product(name='Ноутбук', price=75999.99, stock=12)
True
False
False
False
Ошибка: Цена должна быть положительной


**Задание 6. Абстрактный базовый класс (ABC).**
Создайте `Discount(ABC)` с абстрактным методом `apply(total: float) -> float`. Убедитесь, что попытка создать экземпляр `Discount()` вызывает `TypeError`.

In [10]:
# ============================================================
# ЗАДАНИЕ 6: Абстрактный базовый класс (ABC)
# ============================================================

from abc import ABC, abstractmethod


class Discount(ABC):
    @abstractmethod
    def apply(self, total: float) -> float:
        # контракт: каждая скидка умеет применить себя к сумме
        ...


# ============================================================
# Проверка: Discount() должен падать с TypeError
# ============================================================

try:
    d = Discount()
except TypeError as e:
    print("Ошибка:", e)
    # Ошибка: Can't instantiate abstract class Discount
    #         with abstract method apply

Ошибка: Can't instantiate abstract class Discount without an implementation for abstract method 'apply'


**Задание 7. Полиморфизм.**
Реализуйте наследников `Discount`: `PercentageDiscount(percent)` и `FixedDiscount(amount)`. Убедитесь, что метод `apply` возвращает новую сумму. Проверьте на сумме 1000 руб.

In [11]:
c# ============================================================
# ЗАДАНИЕ 7: Полиморфизм — PercentageDiscount и FixedDiscount
# ============================================================


class PercentageDiscount(Discount):
    def __init__(self, percent: float) -> None:
        if not 0 <= percent <= 100:
            raise ValueError("Процент должен быть в диапазоне 0..100")
        self.percent = percent

    def apply(self, total: float) -> float:
        # возвращаем НОВУЮ сумму, исходный total не меняем
        return total * (1 - self.percent / 100)

    def __repr__(self) -> str:
        return f"PercentageDiscount({self.percent}%)"


class FixedDiscount(Discount):
    def __init__(self, amount: float) -> None:
        if amount < 0:
            raise ValueError("Сумма скидки не может быть отрицательной")
        self.amount = amount

    def apply(self, total: float) -> float:
        # max(0, ...) — итог не уходит в минус
        return max(0, total - self.amount)

    def __repr__(self) -> str:
        return f"FixedDiscount({self.amount})"


# ============================================================
# Проверка на сумме 1000 руб.
# ============================================================

total = 1000.0

discounts = [
    PercentageDiscount(10),   # 10% от 1000 = 100, итого 900
    PercentageDiscount(25),   # 25% от 1000 = 250, итого 750
    FixedDiscount(150),       # 1000 - 150 = 850
    FixedDiscount(2000),      # 1000 - 2000 -> 0 (max защищает)
]

for d in discounts:
    result = d.apply(total)
    print(f"{d!r:25} -> {result:.2f} руб.")

# исходная сумма не изменилась (apply возвращает новую)
print(f"Исходная сумма: {total:.2f} руб.")

PercentageDiscount(10%)   -> 900.00 руб.
PercentageDiscount(25%)   -> 750.00 руб.
FixedDiscount(150)        -> 850.00 руб.
FixedDiscount(2000)       -> 0.00 руб.
Исходная сумма: 1000.00 руб.


**Задание 8. Композиция.**
Создайте класс `Order`, который содержит `customer: Customer`, `items: list[OrderItem]`, `discount: Discount | None`. Метод `total()` возвращает сумму с учётом скидки.

In [14]:
# ============================================================
# ЗАДАНИЕ 8: Композиция — класс Order
# (без изменения OrderItem: считаем subtotal вручную)
# ============================================================

from __future__ import annotations


class Order:
    def __init__(
        self,
        customer: Customer,
        items: list[OrderItem] | None = None,
        discount: Discount | None = None,
    ) -> None:
        # композиция: Order «владеет» клиентом, позициями и скидкой
        self.customer = customer
        self.items: list[OrderItem] = items if items is not None else []
        self.discount = discount

    def add_item(self, item: OrderItem) -> None:
        if not isinstance(item, OrderItem):
            raise TypeError("Ожидается объект OrderItem")
        self.items.append(item)

    def total(self) -> float:
        # считаем subtotal сами: product.price * quantity
        # (не полагаемся на @property subtotal в OrderItem)
        base = sum(i.product.price * i.quantity for i in self.items)
        # применяем скидку, если она есть (полиморфизм из задания 7)
        if self.discount is not None:
            base = self.discount.apply(base)
        return round(base, 2)

    def __len__(self) -> int:
        return len(self.items)

    def __str__(self) -> str:
        return (
            f"Заказ для {self.customer.name}. "
            f"Позиций: {len(self)}. Итого: {self.total():.2f} руб."
        )

    def __repr__(self) -> str:
        return (
            f"Order(customer={self.customer.name!r}, "
            f"items={len(self)}, discount={self.discount!r}, "
            f"total={self.total()!r})"
        )


# ============================================================
# Проверка
# ============================================================

p1 = Product("Ноутбук", 75999.99, 12)
p2 = Product("Мышь", 1999.50, 100)

c = Customer("Иван", 100000.0)

order = Order(c)
order.add_item(OrderItem(p1, 1))
order.add_item(OrderItem(p2, 2))

print(order)                      # Итого: 79999.00 руб.
print(len(order))                 # 2

order.discount = PercentageDiscount(10)
print(order)                      # Итого: 71999.10 руб.

order.discount = FixedDiscount(500)
print(order)                      # Итого: 79499.00 руб.

order.discount = None
print(order)                      # Итого: 79999.00 руб.

print(repr(order))

Заказ для Иван. Позиций: 2. Итого: 79998.99 руб.
2
Заказ для Иван. Позиций: 2. Итого: 71999.09 руб.
Заказ для Иван. Позиций: 2. Итого: 79498.99 руб.
Заказ для Иван. Позиций: 2. Итого: 79998.99 руб.
Order(customer='Иван', items=2, discount=None, total=79998.99)


**Задание 9. Dependency Injection.**
Создайте класс `OrderService`, который принимает в конструкторе `repository` (хранилище заказов). Сервис не должен знать, как именно хранятся заказы — только интерфейс.

In [15]:

from __future__ import annotations

from abc import ABC, abstractmethod


# --- Интерфейс хранилища (абстракция) ---
class OrderRepository(ABC):
    @abstractmethod
    def save(self, order: Order) -> str: ...


# --- Конкретная реализация: хранение в памяти ---
class InMemoryOrderRepository(OrderRepository):
    def __init__(self) -> None:
        self._store: dict[str, Order] = {}
        self._counter: int = 0

    def save(self, order: Order) -> str:
        self._counter += 1
        order_id = f"order-{self._counter}"
        self._store[order_id] = order
        return order_id

    def __repr__(self) -> str:
        return f"InMemoryOrderRepository(size={len(self._store)})"


# --- OrderService: зависит только от интерфейса ---
class OrderService:
    def __init__(self, repository: OrderRepository) -> None:
        # DI: репозиторий приходит снаружи, сервис его не создаёт
        if not isinstance(repository, OrderRepository):
            raise TypeError("repository должен реализовывать OrderRepository")
        self.repository = repository

    def place_order(self, order: Order) -> str:
        # 1) минимальная валидация: заказ должен быть непустым
        if not order.items:
            raise ValueError("Нельзя оформить пустой заказ")
        # 2) сохраняем через интерфейс и возвращаем id
        order_id = self.repository.save(order)
        # 3) возвращаем идентификатор — вызывающий код может его сохранить
        return order_id


# ============================================================
# Проверка (используем классы из заданий 1–8)
# ============================================================

p1 = Product("Ноутбук", 75999.99, 12)
p2 = Product("Мышь", 1999.50, 100)
c = Customer("Иван", 100000.0)

order = Order(c)
order.add_item(OrderItem(p1, 1))
order.add_item(OrderItem(p2, 2))

repo = InMemoryOrderRepository()
service = OrderService(repo)

order_id = service.place_order(order)
print("Сохранён заказ:", order_id)      # Сохранён заказ: order-1
print(repo)                              # InMemoryOrderRepository(size=1)

# пустой заказ не сохраняется
try:
    service.place_order(Order(c))
except ValueError as e:
    print("Ошибка:", e)                 # Ошибка: Нельзя оформить пустой заказ

# неправильный репозиторий отвергается
try:
    OrderService(object())
except TypeError as e:
    print("Ошибка:", e)                 # Ошибка: repository должен реализовывать OrderRepository

Сохранён заказ: order-1
InMemoryOrderRepository(size=1)
Ошибка: Нельзя оформить пустой заказ
Ошибка: repository должен реализовывать OrderRepository


**Задание 10. Принципы SOLID в действии.**
В комментариях к коду (из заданий 8 и 9) расставьте, где применяется каждый принцип: SRP, OCP, LSP, DIP.

In [16]:
# ============================================================
# ЗАДАНИЕ 8: Order
# ============================================================
class Order:
    # SRP: Order отвечает только за состав заказа и расчёт суммы.
    #      Он не сохраняет себя в БД, не печатает счета, не шлёт email.
    # DIP: discount имеет тип Discount (абстракция), а не конкретный
    #      PercentageDiscount/FixedDiscount. Order не знает, какая
    #      именно скидка ему передана.
    # LSP: customer может быть любым Customer, включая VipCustomer —
    #      Order использует только customer.name, что есть у обоих.
    def __init__(
        self,
        customer: Customer,
        items: list[OrderItem] | None = None,
        discount: Discount | None = None,
    ) -> None:
        self.customer = customer
        self.items: list[OrderItem] = items if items is not None else []
        self.discount = discount

    def add_item(self, item: OrderItem) -> None:
        if not isinstance(item, OrderItem):
            raise TypeError("Ожидается объект OrderItem")
        self.items.append(item)

    # OCP: total() не меняется при добавлении новых видов скидок —
    #      он лишь вызывает discount.apply(base) через абстракцию.
    # LSP: любой Discount корректен здесь: PercentageDiscount,
    #      FixedDiscount, будущий SeasonalDiscount и т.д.
    def total(self) -> float:
        base = sum(i.product.price * i.quantity for i in self.items)
        if self.discount is not None:
            base = self.discount.apply(base)
        return round(base, 2)

    def __len__(self) -> int:
        return len(self.items)


# ============================================================
# ЗАДАНИЕ 9: OrderService + репозиторий
# ============================================================

# DIP: OrderService зависит от абстракции OrderRepository,
#      а не от InMemoryOrderRepository. Модуль высокого уровня
#      (сервис) и модуль низкого уровня (хранилище) оба зависят
#      от интерфейса.
class OrderRepository(ABC):
    @abstractmethod
    def save(self, order: Order) -> str: ...


# SRP: InMemoryOrderRepository отвечает только за хранение
#      заказов в памяти. Никакой бизнес-логики.
# OCP: чтобы добавить SQL-хранилище, создаём новый наследник
#      OrderRepository, не меняя OrderService.
# LSP: любой наследник OrderRepository подставляется в
#      OrderService без изменения поведения сервиса.
class InMemoryOrderRepository(OrderRepository):
    def __init__(self) -> None:
        self._store: dict[str, Order] = {}
        self._counter: int = 0

    def save(self, order: Order) -> str:
        self._counter += 1
        order_id = f"order-{self._counter}"
        self._store[order_id] = order
        return order_id


# SRP: OrderService отвечает только за сценарий оформления заказа
#      (валидация + делегирование в репозиторий). Он не хранит
#      заказы сам и не знает, как именно они хранятся.
# DIP: repository приходит снаружи как OrderRepository (абстракция).
#      Сервис не создаёт конкретный репозиторий внутри себя.
# OCP: чтобы поменять хранилище (SQL, Redis, файл), достаточно
#      передать другой OrderRepository — код сервиса не меняется.
class OrderService:
    def __init__(self, repository: OrderRepository) -> None:
        if not isinstance(repository, OrderRepository):
            raise TypeError("repository должен реализовывать OrderRepository")
        self.repository = repository

    def place_order(self, order: Order) -> str:
        if not order.items:
            raise ValueError("Нельзя оформить пустой заказ")
        return self.repository.save(order)


# ============================================================
# Проверка: LSP + DIP в действии
# ============================================================
p1 = Product("Ноутбук", 75999.99, 12)
p2 = Product("Мышь", 1999.50, 100)

# LSP: VipCustomer подставляется туда, где ожидается Customer
customer: Customer = VipCustomer("Мария", 1000.0, bonus=500.0)

order = Order(customer)                   # Order принимает Customer
order.add_item(OrderItem(p1, 1))
order.add_item(OrderItem(p2, 2))

# LSP: любая скидка подставляется в Order
order.discount = PercentageDiscount(10)
print(order.total())                      # 71999.1

# DIP: OrderService не знает, какой репозиторий ему передали
repo: OrderRepository = InMemoryOrderRepository()
service = OrderService(repo)
order_id = service.place_order(order)
print(order_id)                       

71999.09
order-1


---
### Задания по прикладной задаче (11–20)

**Прикладная задача:** «Доменная модель интернет-магазина».
Требуется собрать систему, в которой клиенты могут делать заказы, товары имеют остатки на складе, к заказам применяются скидки, а сами заказы сохраняются в хранилище.

**Задание 11. Настройка окружения и валидация Product**
Инициализируйте папку `lab4` в репозитории `python-labs`. Создайте ноутбук. Реализуйте класс `Product` с валидацией в `@property` (цена и остаток не могут быть отрицательными).

In [16]:
!git clone https://gitlab.com/<логин>/python-labs.git
%cd python-labs
!mkdir -p lab4
%cd lab4
!git config --global user.name "Ваше Имя"
!git config --global user.email "ваш@email.com"

/bin/bash: line 1: логин: No such file or directory
[Errno 2] No such file or directory: 'python-labs'
/content
/content/lab4


In [17]:
class Product:
    def __init__(self, name: str, price: float, stock: int) -> None:
        self.name = name
        self.price = price
        self.stock = stock

    @property
    def price(self) -> float:
        return self._price

    @price.setter
    def price(self, value: float) -> None:
        # Ваш код
        ...

    @property
    def stock(self) -> int:
        return self._stock

    @stock.setter
    def stock(self, value: int) -> None:
        # Ваш код
        ...

**Задание 12. Иерархия клиентов**
Реализуйте класс `Customer` (с балансом и методами `deposit/withdraw`) и `VipCustomer` (с бонусом при создании и скидкой 10%). Обрабатывайте ошибку при недостатке средств.

In [18]:
class Customer:
    # Ваш код
    ...

class VipCustomer(Customer):
    # Ваш код
    ...

**Задание 13. Позиция заказа и сам заказ**
Реализуйте `OrderItem` (товар, количество, подсчёт `subtotal`). Реализуйте класс `Order`, поддерживающий добавление позиций, подсчёт суммы, применение скидки, вывод через `__str__` и подсчёт количества позиций через `__len__`.

In [19]:
class OrderItem:
    # Ваш код
    ...

class Order:
    # Ваш код
    ...

**Задание 14. Иерархия скидок**
Создайте `Discount(ABC)`, `PercentageDiscount`, `FixedDiscount` и `NoDiscount`. Убедитесь, что `Order.total()` корректно работает с любой из них.

In [20]:
# Ваш код
...

**Задание 15. Абстрактный репозиторий (DIP)**
Создайте `OrderRepository(ABC)` с методами `save(order)`, `get_by_id(order_id)`, `get_all()`.

In [21]:
# Ваш код
...

**Задание 16. Реализация in-memory хранилища**
Создайте `InMemoryOrderRepository(OrderRepository)`, который хранит заказы в словаре. Для генерации ID используйте модуль `uuid`.

In [22]:
# Ваш код
...

**Задание 17. Сервис заказов**
Реализуйте `OrderService`, который принимает репозиторий и предоставляет метод `place_order(order)` для сохранения.

In [23]:
# Ваш код
...

**Задание 18. Сборка сценария (Main)**
Напишите функцию `main()`, которая:
1. Создаёт 2-3 товара.
2. Создаёт клиента и пополняет его баланс.
3. Формирует заказ с несколькими позициями.
4. Применяет скидку.
5. Сохраняет заказ через `OrderService`.
6. Печатает чек (вывод заказа) и сообщает ID сохранённого заказа.

In [24]:
def main() -> None:
    # Ваш код
    ...

main()

**Задание 19. Обработка ошибок в домене**
Добавьте проверку: если товара не хватает на складе (`product.stock < quantity`), при добавлении в заказ должно выбрасываться исключение `ValueError`. Обработайте его в `main()`.

In [25]:
# Ваш код
...

**Задание 20. Комплексный отчёт и сдача**
1. Добавьте docstring ко всем классам и методам.
2. Проверьте код линтером (PEP 8).
3. Создайте `README.md`: опишите архитектуру классов и обоснуйте применение принципов SOLID (особенно DIP и OCP).
4. Сделайте коммит и `git push` в GitLab.

In [26]:
%%writefile README.md
# Лабораторная работа № 4

## Тема
Объектно-ориентированное программирование. Принципы SOLID.

## Архитектура
- `Product` — товар, инкапсуляция цены и остатка.
- `Customer` — клиент с балансом, `VipCustomer` — скидка 10%.
- `OrderItem` — позиция заказа, магические методы `__eq__`, `__add__`.
- `Order` — заказ, композиция позиций и скидки.
- `Discount` (ABC) — абстракция скидки, реализации `PercentageDiscount`, `FixedDiscount`.
- `OrderRepository` (ABC) — интерфейс хранилища, `InMemoryOrderRepository` — реализация.
- `OrderService` — оркестратор, зависит от `OrderRepository` (DIP).

## Принципы SOLID
- **SRP:** `Order` считает сумму, `Discount` — скидку, `Repository` — хранит.
- **OCP:** новая скидка реализуется без изменения `Order`.
- **LSP:** любой наследник `Discount` подставляется в `Order`.
- **DIP:** `OrderService` зависит от абстракции, а не от `InMemoryOrderRepository`.

## Ссылка на репозиторий
https://gitlab.com/<логин>/python-labs

Writing README.md


In [27]:
!git add .
!git commit -m "ЛР4: ООП, SOLID, доменная модель магазина"
!git push origin main

fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git


## Контрольные вопросы

1. Чем `__str__` отличается от `__repr__`?
2. Зачем нужен `@property`, если можно просто сделать атрибут публичным?
3. Что такое `@classmethod` и `@staticmethod`? В чём разница?
4. Как `super()` работает в цепочке наследования?
5. Что такое абстрактный класс и зачем нужны абстрактные методы?
6. Как перегрузка операторов (магические методы) помогает сделать код читаемее?
7. Объясните принцип Open/Closed на примере иерархии скидок.
8. Что такое Dependency Injection и почему это лучше, чем создавать объекты внутри класса?
9. Как принцип подстановки Лисков (LSP) связан с исключениями в наследниках?
10. Почему считается, что наследование увеличивает связанность кода, и как композиция помогает это исправить?

## Дополнительно

1. Оформите выполненные задания в виде Jupyter-ноутбука `lab4_oop.ipynb`.
2. Убедитесь, что ноутбук выполняется сверху вниз без ошибок (Kernel → Restart & Run All).
3. В репозитории `python-labs` создайте папку `lab4` и добавьте туда ноутбук и `README.md`.
4. Создайте `.gitignore` (исключите `__pycache__/`, `.ipynb_checkpoints/`).
5. Зафиксируйте изменения:
```bash
git add .
git commit -m "ЛР4: ООП, SOLID, доменная модель магазина"
git push origin main
```
6. В конце `README.md` добавьте раздел «Ссылка на репозиторий» и укажите URL.
7. Прикрепите ссылку и ноутбук в системе дистанционного обучения.